In [1]:
import matplotlib.pyplot as plt

from spectrum.spectrum2D import convNu2Ene, Spectrum2D
%matplotlib inline
import numpy as np
# np.set_printoptions(precision=4)

import sys
sys.path.append("/home/vlew/CQCParse")

from CQCParse.parsing import GaussianDataParser
from CQCParse.relay import DataVault

from wilson import rendering
from wilson import analysis

from wilson.utils import get_package_root

wilson_root = get_package_root()
# data_vault = DataVault(wilson_root+'/tests/test_database/mini_files_database.csv')
data_vault = DataVault('/mnt/c/Users/vle014/OneDrive - UiT Office 365/Documents/files_fram/files_database.csv')

dataframe_gaussian = data_vault.getting_files_DB("gaussian")
method_basis = dataframe_gaussian[(dataframe_gaussian['code'] == 'ACAC') & (dataframe_gaussian['method'] != 'PBE0')][["code", "method", "basis_set"]]
tuples_method_basis = [(row['code'], row['method'], row['basis_set']) for index, row in method_basis.iterrows()]
# tuples_method_basis

# omega1 = np.arange(*regions[region][0])
# omega2 = np.arange(*regions[region][1])
# omega1 = [1164., 1175., 1186., 1314., 1324., 1334., 1362., 1372., 1382., 1776., 1786., 1796.]
# omega2 = [1736., 1746., 1757., 1975., 1986., 1997., 2295., 2301., 2311., 2323., 2931., 2942., 2952., 3073., 3083., 3093.]

region = 1
regions = {1: ((1280., 3150., 235.41), (1589., 6050., 465.41)) }
omega1 = np.arange(*regions[region][0])
omega2 = np.arange(*regions[region][1])

print(omega1)
print(omega2)

vibEL = False
datain = data_vault.make_DatainputDict('gaussian', ('ACAC', 'B3LYP', 'cc_pVQZ'), '')

dictInputs = {'parserObject': GaussianDataParser(datain), 'el_terms_select': [0,1], 'mech_terms_select': [0,1]}
spectrumObj = Spectrum2D(omega1, omega2)
spectrumObj.load_data(dictInputs['parserObject'])
spectrumObj.setSpectrumSettings(Gamma_rc=10., diag_margin_rc=10., vib_levels_harmonic=vibEL)
# spectrumObj.conversion2InternalUnits() # need now at least for diag margin_rs in addTerms()
spectrumObj.addTerms(dictInputs['el_terms_select'], dictInputs['mech_terms_select'])
spectrumObj.precalculateParts()

print('spectrumObj.mech_avrg_tensors')
for i in spectrumObj.mech_avrg_tensors:
    print(repr(i), '\n')
    
print('before computedSpectrum.fundamentals')
print(spectrumObj.fundamentals)
print(sorted(list(spectrumObj.fundamentals.values())))
print('\ncomputedSpectrum.all_states\n', spectrumObj.all_states)

# 'mu_Q', 'mu_QQ', 'alpha_Q', 'alpha_QQ', 'F_abc'
# spectrumObj.deriv_data['mu_Q'].shape
# spectrumObj.deriv_data['mu_QQ'].shape
# spectrumObj.deriv_data['alpha_Q'].shape
# spectrumObj.deriv_data['alpha_QQ'].shape

sec_hypol_dataALL = 0
sec_hypol_data1 = 0

if dictInputs['mech_terms_select']:
    electrical, Qab_contrib_dict = spectrumObj.intensity_electrical()
    sec_hypol_data1 += electrical
    print(f'\nElectrical {dictInputs["el_terms_select"]}')
    print(repr(sec_hypol_data1))

sec_hypol_dataALL+=sec_hypol_data1

sec_hypol_data2 = 0
if dictInputs['mech_terms_select']:
    mechall, Qabc_contrib_dict = spectrumObj.intensity_mechanical()
    sec_hypol_data2 += mechall
    print(f'Mechanical {dictInputs["mech_terms_select"]}')
    print(repr(sec_hypol_data2))

sec_hypol_dataALL+=sec_hypol_data2

intensity = abs(sec_hypol_dataALL) ** 2

print('\nSum both')
print(repr(sec_hypol_dataALL))

print('\nIntensity abs(sec_hypol_dataALL)**2')
print(repr(intensity))

/home/vlew/miniconda3/envs/wilsonstuff/lib/python3.10/site-packages/numpy/_core/getlimits.py:548: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


[1280.              1515.41            1750.8200000000002 1986.2300000000002 2221.6400000000003 2457.05            2692.4600000000005 2927.870000000001 ]
[1589.              2054.41            2519.8199999999997 2985.2299999999996 3450.6399999999994 3916.0499999999993 4381.459999999999  4846.869999999999  5312.279999999999  5777.689999999999 ]

Used vibrational energy levels are harmonic? - False
collectionFreqRes
 3 {('b,a', 'zero,a'), ('a+b,a', 'zero,a'), ('c,a', 'zero,a')}
collectionFreqDiff
 4 {'a+b+c,zero', 'b+c,a', 'c,a+b', 'a+b,c'}
collectionAveraging
 3 {(('mu_Q', ('a',)), ('alpha_Q', ('b',)), ('mu_Q', ('c',))), (('mu_Q', ('a',)), ('alpha_QQ', ('a', 'b')), ('mu_Q', ('b',))), (('mu_Q', ('a',)), ('alpha_Q', ('b',)), ('mu_QQ', ('a', 'b')))}
('a+b,a', 'zero,a') None
('b,a', 'zero,a') None
('a+b,a', 'zero,a') ('a+b+c,zero', 'c,a+b')
('c,a', 'zero,a') ('a+b,c', 'b+c,a')
spectrumObj.mech_avrg_tensors
array([[[-1.6293040201152202e-06, -2.7516632956318587e-07, -1.9063712613206133e-07, .